## Cell 1: Setup
**What this demonstrates**: CRAG requires three API integrations — Anthropic for grading and generation, OpenAI for embeddings, and Tavily for web search fallback. If `TAVILY_API_KEY` is absent, the notebook uses a mock web result so the full pipeline can still be demonstrated.
**Expected output**: ✓ Setup complete with model names, Tavily mode (live or mock), and masked key suffixes.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv tavily-python

# ── Standard library ─────────────────────────────────────────────────────────
import os
import time
import pathlib
from dataclasses import dataclass, field

# ── Third-party ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
_tavily_key    = os.environ.get('TAVILY_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'
# Tavily is optional — notebook degrades to mock web results if absent
TAVILY_LIVE = bool(_tavily_key)

# ── Model constants ───────────────────────────────────────────────────────────
# Haiku grades each chunk (fast 1-5 scoring, not a reasoning task)
GRADE_MODEL  = 'claude-haiku-4-5-20251001'
# Sonnet generates the final answer (reasoning quality matters)
ANSWER_MODEL = 'claude-sonnet-4-6'
EMBED_MODEL  = 'text-embedding-3-small'
CHROMA_DIR   = './chroma_crag'
CHUNK_SIZE   = 400

# Web search fallback threshold — if average chunk relevance score < this, trigger web search
RELEVANCE_THRESHOLD = 3.0  # scale: 1 (not relevant) to 5 (highly relevant)

# ── Clients ───────────────────────────────────────────────────────────────────
client:     Anthropic        = Anthropic()
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# ── Tavily web search (live or mock) ─────────────────────────────────────────
if TAVILY_LIVE:
    from tavily import TavilyClient
    tavily_client = TavilyClient(api_key=_tavily_key)


def web_search(query: str, max_results: int = 3) -> list[dict]:
    """Search the web for current information using Tavily.

    Falls back to a mock result if TAVILY_API_KEY is not set, so the
    full CRAG pipeline can be demonstrated without a live API key.

    Args:
        query:       The search query (typically the user's original question).
        max_results: Number of web results to return.

    Returns:
        List of dicts with 'url', 'title', 'content' keys.
    """
    if TAVILY_LIVE:
        response = tavily_client.search(query=query, max_results=max_results)
        return response.get('results', [])
    # ── Mock result for demo environments without Tavily key ──────────────────
    # Simulates what Tavily would return for a Fed rate query
    return [
        {
            'url':     'https://federalreserve.gov/newsevents/pressreleases/monetary20241218.htm',
            'title':  'Federal Reserve issues FOMC statement',
            'content': (
                'The Federal Open Market Committee decided to lower the target range for the '
                'federal funds rate by 1/4 percentage point to 4-1/4 to 4-1/2 percent. '
                'The Committee seeks to achieve maximum employment and inflation at the rate '
                'of 2 percent over the longer run. Recent indicators suggest that economic '
                'activity has continued to expand at a solid pace. The unemployment rate has '
                'moved up but remains low. Inflation has made progress toward the 2 percent '
                'objective but remains somewhat elevated. In support of its goals, the '
                'Committee decided to lower the target range for the federal funds rate.'
            ),
        },
        {
            'url':     'https://reuters.com/markets/us/fed-cuts-rates-december-2024',
            'title':  'Fed cuts rates by quarter point in December 2024',
            'content': (
                'The U.S. Federal Reserve cut interest rates by 25 basis points on Wednesday, '
                'its third consecutive reduction, bringing the federal funds rate to a target '
                'range of 4.25% to 4.50%. Fed Chair Jerome Powell signalled a more cautious '
                'approach to further cuts in 2025, citing persistent inflation and a resilient '
                'labour market. Markets had fully priced in the 25bp cut.'
            ),
        },
    ]


print('✓ Setup complete')
print(f'  Grade model:   {GRADE_MODEL}')
print(f'  Answer model:  {ANSWER_MODEL}')
print(f'  Embed model:   {EMBED_MODEL}')
print(f'  Anthropic key: ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:    ...{_openai_key[-4:]}')
print(f'  Tavily:        {"LIVE (" + _tavily_key[-4:] + ")" if TAVILY_LIVE else "MOCK — no key set, using simulated web results"}')
print(f'  Relevance threshold: {RELEVANCE_THRESHOLD} / 5  (fallback fires if avg score below this)')

## Cell 2: Data — Internal Lending Policy (Intentionally Outdated)
**What this demonstrates**: The internal corpus is a consumer lending policy document — a static internal document with no market data or macroeconomic content. A query about Federal Reserve rate decisions will retrieve topically adjacent chunks (interest rates appear in the policy's loan pricing sections) but none will score high on relevance grading. This is the designed CRAG test case: plausible retrieval, low actual relevance.
**Expected output**: Chunk count, a preview of the interest-rate content that exists in the policy (showing why cosine similarity retrieves it), and a note on why it will fail the relevance grade.

In [ ]:
# ── Locate document ───────────────────────────────────────────────────────────
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data.'

doc_text = (BASE_DIR / 'fintech_policy.txt').read_text(encoding='utf-8')

# ── Chunk and index ───────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    separators=['\n================================================================================\n', '\n\n', '\n', ' '],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=40,
)
raw_chunks = splitter.split_text(doc_text)
all_docs: list[Document] = [
    Document(page_content=chunk, metadata={'chunk_idx': i, 'source': 'Meridian Bank Lending Policy'})
    for i, chunk in enumerate(raw_chunks)
]

vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name='crag_internal',
    persist_directory=CHROMA_DIR,
)

print(f'Internal corpus: fintech_policy.txt  ({len(doc_text):,} chars  →  {len(all_docs)} chunks)')
print(f'Source:          Meridian Bank Consumer Lending Policy (static internal document)')
print()

# Show interest-rate content that exists in the policy
# (demonstrates why cosine search retrieves it, and why the grader rejects it)
rate_chunks = [d for d in all_docs if 'rate' in d.page_content.lower() or 'interest' in d.page_content.lower()]
print(f'Chunks mentioning "rate" or "interest": {len(rate_chunks)}')
print()
print('Sample — what the policy says about interest rates:')
if rate_chunks:
    print(f'  {rate_chunks[0].page_content[:200].replace(chr(10), " ")}...')
print()
print('Why CRAG triggers fallback on a Fed rate query:')
print('  Cosine similarity: policy mentions rates → chunks retrieved ✓')
print('  Relevance grading: policy describes loan pricing, not Fed decisions → score < 3 ✗')
print('  CRAG response:     web search → current FOMC announcement')

## Cell 3: Core — CRAG Pipeline
**What this demonstrates**: Three functions implement CRAG: `grade_chunk` (LLM scores relevance 1–5), `refine_web_results` (extracts factual content from raw web text), and `corrective_rag` (retrieve → grade → route → generate). The routing decision is deterministic on the average relevance score against the threshold.
**Expected output**: Function definitions confirmed, then a test of `grade_chunk` on one relevant and one irrelevant chunk for the Fed rate query.

In [ ]:
# ── Grade: score a chunk's relevance 1-5 ─────────────────────────────────────
def grade_chunk(query: str, chunk: Document) -> tuple[int, str]:
    """Score how relevant a retrieved chunk is to the query on a 1–5 scale.

    Scale:
        5 — directly answers the question with specific facts/figures
        4 — strongly relevant, contains most of what is needed
        3 — partially relevant, tangentially related
        2 — low relevance, same domain but different topic
        1 — not relevant to the query

    Uses Haiku for speed — this is a scoring task, not a reasoning task.
    One call per retrieved chunk; with k=5, this is 5 fast parallel-able calls.

    Args:
        query: The user's question.
        chunk: A retrieved Document.

    Returns:
        (score 1–5, one-sentence rationale)
    """
    response = client.messages.create(
        model=GRADE_MODEL,
        max_tokens=80,
        system=(
            'Rate how relevant this document excerpt is to the question on a scale of 1–5. '
            '5: directly answers the question with specific facts or figures. '
            '4: strongly relevant — contains most of what is needed. '
            '3: partially relevant — same domain, tangentially related. '
            '2: low relevance — same broad domain, different topic. '
            '1: not relevant. '
            'Respond in exactly this format:\n'
            'Score: 1 or 2 or 3 or 4 or 5\n'
            'Reason: one sentence'
        ),
        messages=[{'role': 'user', 'content': (
            f'Question: {query}\n\n'
            f'Excerpt:\n{chunk.page_content[:500]}'
        )}],
    )
    text  = response.content[0].text.strip()
    score = 3  # default if parse fails
    for line in text.split('\n'):
        if line.startswith('Score:'):
            try:
                score = int(line.split(':', 1)[1].strip())
            except ValueError:
                pass
    reason = text.split('Reason:', 1)[-1].strip() if 'Reason:' in text else text
    return score, reason


# ── Refine: extract factual content from raw web results ─────────────────────
def refine_web_results(query: str, web_results: list[dict]) -> str:
    """Extract factual content relevant to the query from raw web search results.

    Raw web results contain navigation text, ads, and boilerplate that degrade
    generation quality. This step calls the LLM to extract only the factual
    content relevant to the query before it enters the generation context.

    Args:
        query:       The user's original question.
        web_results: Raw results from web_search() — list of dicts with 'content'.

    Returns:
        A cleaned string of factual content ready for the generation prompt.
    """
    raw = '\n\n---\n\n'.join(
        f'[Source: {r["title"]} | {r["url"]}]\n{r["content"]}'
        for r in web_results
    )
    response = client.messages.create(
        model=GRADE_MODEL,  # Haiku: extraction task, not reasoning
        max_tokens=400,
        system=(
            'Extract the factual information from the web search results that is '
            'relevant to answering the question. '
            'Keep specific figures, dates, and attributions. '
            'Remove navigation text, promotional content, and unrelated context. '
            'Preserve source attribution (title and URL) for each fact extracted.'
        ),
        messages=[{'role': 'user', 'content': f'Question: {query}\n\nWeb results:\n{raw}'}],
    )
    return response.content[0].text.strip()


# ── CRAG: full corrective pipeline ───────────────────────────────────────────
def corrective_rag(query: str, k: int = 5) -> dict:
    """Run the CRAG pipeline: retrieve → grade → route → generate.

    Routing logic:
        avg_score >= RELEVANCE_THRESHOLD  → use internal corpus (Correct path)
        avg_score < RELEVANCE_THRESHOLD   → web search + refine (Incorrect path)

    Args:
        query: The user's question.
        k:     Number of chunks to retrieve from the internal corpus.

    Returns:
        dict with: answer, retrieved_chunks, grades, avg_score, path_taken,
        web_results (if fallback fired), refined_context, total_ms
    """
    t_start = time.perf_counter()

    # ── Step 1: Retrieve ──────────────────────────────────────────────────────
    retrieved = vectorstore.similarity_search(query, k=k)

    # ── Step 2: Grade each chunk ──────────────────────────────────────────────
    grades: list[tuple[Document, int, str]] = []
    for chunk in retrieved:
        score, reason = grade_chunk(query, chunk)
        grades.append((chunk, score, reason))

    avg_score = sum(s for _, s, _ in grades) / len(grades) if grades else 0.0

    # ── Step 3: Route based on average score ─────────────────────────────────
    web_results: list[dict] = []
    refined_context: str    = ''

    if avg_score >= RELEVANCE_THRESHOLD:
        # Correct path: internal corpus is good — strip low-scoring chunks
        path_taken    = 'internal'
        good_chunks   = [doc for doc, score, _ in grades if score >= RELEVANCE_THRESHOLD]
        # If filtering leaves nothing, use all retrieved (score was marginal)
        context_docs  = good_chunks if good_chunks else retrieved
        generation_context = '\n\n---\n\n'.join(
            f'[Internal doc, chunk {i+1}]\n{doc.page_content}'
            for i, doc in enumerate(context_docs)
        )
    else:
        # Incorrect path: internal corpus insufficient → web search fallback
        path_taken     = 'web_fallback'
        web_results    = web_search(query)
        refined_context = refine_web_results(query, web_results)
        generation_context = refined_context

    # ── Step 4: Generate ──────────────────────────────────────────────────────
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=400,
        system=(
            'You are a financial information analyst. '
            'Answer the question using ONLY the provided context. '
            'Cite the source (internal document or web URL) for each specific fact. '
            'If the context does not contain enough information to answer, say so explicitly.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{generation_context}\n\nQuestion: {query}'}],
    )

    return {
        'answer':             response.content[0].text.strip(),
        'retrieved_chunks':   retrieved,
        'grades':             grades,
        'avg_score':          avg_score,
        'path_taken':         path_taken,
        'web_results':        web_results,
        'refined_context':    refined_context,
        'total_ms':           (time.perf_counter() - t_start) * 1000,
    }


# ── Smoke test: grade_chunk on one relevant and one irrelevant chunk ───────────
print('Smoke test — grade_chunk for Fed rate query:')
print()
test_query = 'What was announced about the Federal Reserve interest rate last week?'

# A chunk about loan interest rates — will score low (same domain, wrong topic)
test_chunk = next((d for d in all_docs if 'rate' in d.page_content.lower()), None)
if test_chunk:
    score, reason = grade_chunk(test_query, test_chunk)
    bar = '█' * score + '░' * (5 - score)
    print(f'  Internal policy chunk (loan rates):')
    print(f'  Score: {score}/5  [{bar}]')
    print(f'  Reason: {reason}')
    print(f'  Content: {test_chunk.page_content[:80].replace(chr(10), " ")}...')
    print()
    print(f'  → avg score below {RELEVANCE_THRESHOLD}? {score < RELEVANCE_THRESHOLD} → web fallback would fire')

## Cell 4: Run — Federal Reserve Rate Query
**What this demonstrates**: End-to-end CRAG on a query not covered by the internal corpus — `"What was announced about the Federal Reserve interest rate last week?"`. All five retrieved internal chunks grade low, avg_score falls below threshold, web search fires, results are refined, and the final answer is generated from web content.
**Expected output**: Per-chunk grades with scores and reasons, the routing decision, the web results (live or mock), and the final grounded answer with source attribution.

In [ ]:
QUERY = 'What was announced about the Federal Reserve interest rate last week?'

print(f'Query: {QUERY!r}')
print()

result = corrective_rag(QUERY, k=5)

# ── Grading results ───────────────────────────────────────────────────────────
print(f'Step 1: Retrieved {len(result["retrieved_chunks"])} chunks from internal corpus')
print(f'Step 2: Relevance grading (threshold: {RELEVANCE_THRESHOLD}/5)')
print()
for i, (doc, score, reason) in enumerate(result['grades'], 1):
    bar    = '█' * score + '░' * (5 - score)
    status = '✓' if score >= RELEVANCE_THRESHOLD else '✗'
    preview = doc.page_content[:55].replace('\n', ' ')
    print(f'  [{i}] {status} Score: {score}/5  [{bar}]  {preview!r}...')
    print(f'       {reason}')
    print()

print(f'  Average score: {result["avg_score"]:.2f} / 5')
print()

# ── Routing decision ──────────────────────────────────────────────────────────
print('Step 3: Routing decision')
if result['path_taken'] == 'web_fallback':
    print(f'  avg_score ({result["avg_score"]:.2f}) < threshold ({RELEVANCE_THRESHOLD})')
    print(f'  → WEB FALLBACK fired  ({'Tavily live' if TAVILY_LIVE else 'mock result'})')
    print()
    print(f'  Web results returned: {len(result["web_results"])}')
    for r in result['web_results']:
        print(f'    • {r["title"]} — {r["url"]}')
else:
    print(f'  avg_score ({result["avg_score"]:.2f}) >= threshold ({RELEVANCE_THRESHOLD})')
    print('  → INTERNAL PATH — using corpus results')

# ── Final answer ──────────────────────────────────────────────────────────────
print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(result['answer'])
print('=' * 65)
print()
print(f'Path: {result["path_taken"]}  |  Total: {result["total_ms"]:.0f} ms')

## Cell 5: Inspect — Grading Scores and Web Search Trigger
**What this demonstrates**: Expose the full CRAG decision trace — show the score distribution across all retrieved chunks, the raw web search results before refinement, the refined context after knowledge extraction, and a side-by-side comparison of what the internal corpus would have produced vs what the web fallback produced.
**Expected output**: Score distribution bar chart (ASCII), raw vs refined web content comparison, and a baseline answer from internal corpus showing why the fallback improved the result.

In [ ]:
# ── Score distribution ────────────────────────────────────────────────────────
print('── GRADING SCORE DISTRIBUTION ───────────────────────────────────────────────')
scores = [score for _, score, _ in result['grades']]
for score_val in range(5, 0, -1):
    count = scores.count(score_val)
    bar   = '█' * count
    print(f'  {score_val}/5  {bar:<5} ({count} chunk{"s" if count != 1 else ""})')
print()
print(f'  Average: {result["avg_score"]:.2f}  |  Threshold: {RELEVANCE_THRESHOLD}')
print(f'  Decision: {"FALLBACK" if result["path_taken"] == "web_fallback" else "INTERNAL"}')

# ── Raw web result vs refined output ─────────────────────────────────────────
if result['path_taken'] == 'web_fallback' and result['web_results']:
    print()
    print('── KNOWLEDGE REFINEMENT: RAW → REFINED ──────────────────────────────────────')
    raw_sample = result['web_results'][0]['content']
    print('  Raw web content (first result, first 300 chars):')
    print(f'  {raw_sample[:300].replace(chr(10), " ")}...')
    print()
    print('  Refined context (factual extraction):')
    print(f'  {result["refined_context"][:400].replace(chr(10), " ")}...')
    print()
    print('  Refinement removes: navigation text, boilerplate, unrelated context.')
    print('  Refinement keeps:   specific figures, dates, source attribution.')

# ── Baseline comparison: what internal retrieval would have produced ───────────
print()
print('── BASELINE: INTERNAL RETRIEVAL WITHOUT CRAG ────────────────────────────────')
print('What would plain RAG have generated from the internal corpus?')
print()

# Generate from internal docs without grading — shows what CRAG corrected
baseline_context = '\n\n---\n\n'.join(
    f'[Internal chunk {i+1}]\n{doc.page_content}'
    for i, doc in enumerate(result['retrieved_chunks'])
)
baseline_resp = client.messages.create(
    model=ANSWER_MODEL,
    max_tokens=200,
    system=(
        'You are a financial analyst. Answer using only the provided context. '
        'If the context does not contain the answer, say so clearly.'
    ),
    messages=[{'role': 'user', 'content': f'Context:\n{baseline_context}\n\nQuestion: {QUERY}'}],
)
print('  Plain RAG answer (no grading, no web fallback):')
print(f'  {baseline_resp.content[0].text.strip()[:300]}')
print()
print('  CRAG answer (with web fallback):')
print(f'  {result["answer"][:300]}')
print()
print('The CRAG answer references specific figures and sources.')
print('The baseline acknowledges it cannot answer — CRAG corrected this automatically.')

## Cell 6: Fintech — Market News Q&A with Web Fallback
**What this demonstrates**: CRAG on a second market-news query — `"What happened to major bank stock prices after the latest Federal Reserve meeting?"` — and a contrast query that stays on the internal path (`"What is the minimum credit score required for a personal loan?"`) to show both routing paths in a single cell.
**Expected output**: Both queries run with grading scores and routing decisions visible, demonstrating CRAG correctly routing the market query to web search and the policy query to internal retrieval.

In [ ]:
# Two queries: one that fires web fallback, one that stays internal
# This demonstrates CRAG's routing decision on each
MARKET_QUERY = 'What happened to major bank stock prices after the latest Federal Reserve meeting?'
POLICY_QUERY = 'What is the minimum credit score required for a personal loan?'

queries = [
    ('Market news (expects web fallback)',  MARKET_QUERY),
    ('Policy lookup (expects internal)',    POLICY_QUERY),
]

for label, query in queries:
    print(f'── {label.upper()} ──────────────────────────────────────────────────────')
    print(f'   Query: {query!r}')
    print()

    t0  = time.perf_counter()
    res = corrective_rag(query, k=5)
    ms  = (time.perf_counter() - t0) * 1000

    # Print score summary for each chunk
    for i, (doc, score, reason) in enumerate(res['grades'], 1):
        bar = '█' * score + '░' * (5 - score)
        print(f'   [{i}] {score}/5 [{bar}]  {reason[:65]}')

    print()
    path_label = 'WEB FALLBACK ↗' if res['path_taken'] == 'web_fallback' else 'INTERNAL CORPUS ✓'
    print(f'   avg_score: {res["avg_score"]:.2f}  →  {path_label}')
    print()
    print('   ANSWER:')
    # Print first 350 chars of answer
    answer_preview = res['answer'][:350].replace('\n', '\n   ')
    print(f'   {answer_preview}')
    if len(res['answer']) > 350:
        print('   ...')
    print()
    print(f'   [{ms:.0f} ms  |  path: {res["path_taken"]}]')
    print()

print('─' * 65)
print('CRAG routing summary:')
print(f'  Market news query  → scores low on internal policy doc → web fallback')
print(f'  Policy lookup      → scores high on internal policy doc → internal path')
print()
print('In production: log path_taken per query.')
print('A rising web_fallback rate signals the internal corpus needs updating.')